In [2]:
# 1. IMPORT LIBRARIES

import pandas as pd
import numpy as np

# 2. LOAD DATA

df = pd.read_csv(r"C:\Users\prafu\OneDrive\Documents\Zomato_KPT\zomato_kpt_dataset.csv")

print("Shape:", df.shape)
df.head()

Shape: (550502, 17)


,order_time,restaurant_id,cuisine_type,restaurant_capacity,restaurant_rating,chef_count,efficiency_score,total_items,item_complexity_score,active_orders_in_kitchen,queue_length,kitchen_utilization,weather,festival_flag,hour,weekend,prep_time
0,2024-01-01 00:00:00,37,Biryani,10,3.6,5,0.876097,4,6.24,0,0.0,0.0,Rainy,1,0,0,25.77
1,2024-01-01 00:00:00,198,Continental,13,3.0,8,0.812020,6,14.66,0,0.0,0.0,Rainy,1,0,0,55.89
2,2024-01-01 00:00:00,50,Fast Food,5,4.5,3,0.994893,4,4.46,0,0.0,0.0,Rainy,1,0,0,8.59
3,2024-01-01 00:00:00,187,Biryani,7,4.1,4,0.978308,5,11.80,0,0.0,0.0,Rainy,1,0,0,33.36
4,2024-01-01 00:00:00,129,North Indian,5,4.1,3,0.980710,2,3.63,0,0.0,0.0,Rainy,1,0,0,24.25


In [3]:
# Check missing values
print("\nMissing Values:\n")
print(df.isnull().sum())

# Basic stats
print("\nBasic Description:\n")
print(df.describe())


Missing Values:

order_time                  0
restaurant_id               0
cuisine_type                0
restaurant_capacity         0
restaurant_rating           0
chef_count                  0
efficiency_score            0
total_items                 0
item_complexity_score       0
active_orders_in_kitchen    0
queue_length                0
kitchen_utilization         0
weather                     0
festival_flag               0
hour                        0
weekend                     0
prep_time                   0
dtype: int64

Basic Description:

       restaurant_id  restaurant_capacity  restaurant_rating     chef_count  \
count  550502.000000        550502.000000      550502.000000  550502.000000   
mean      100.682426            10.882722           4.270943       5.995388   
std        56.128945             4.249872           0.558938       2.113678   
min         0.000000             4.000000           3.000000       3.000000   
25%        52.000000             7.000000  

In [4]:
# Drop raw order_time
df = df.drop(columns=["order_time"])

In [5]:
# Create restaurant average prep time feature

restaurant_avg = df.groupby("restaurant_id")["prep_time"].mean()

df["restaurant_avg_prep"] = df["restaurant_id"].map(restaurant_avg)

# Drop restaurant_id (avoid leakage & overfitting)
df = df.drop(columns=["restaurant_id"])

In [6]:
# Cyclical encoding for hour

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Drop raw hour
df = df.drop(columns=["hour"])

In [7]:
# Interaction between congestion and complexity

df["congestion_complexity"] = (
    df["queue_length"] * df["item_complexity_score"]
)

In [8]:
# Log transform queue length (stabilizes heavy tail)

df["log_queue"] = np.log1p(df["queue_length"])

In [9]:
df = df.drop(columns=["active_orders_in_kitchen"])

In [11]:
from sklearn.preprocessing import LabelEncoder

le_cuisine = LabelEncoder()
le_weather = LabelEncoder()

df["cuisine_type"] = le_cuisine.fit_transform(df["cuisine_type"])
df["weather"] = le_weather.fit_transform(df["weather"])

In [12]:
print("Final Columns:\n")
print(df.columns)

print("\nFinal Shape:", df.shape)
df.head()

Final Columns:

Index(['cuisine_type', 'restaurant_capacity', 'restaurant_rating',
       'chef_count', 'efficiency_score', 'total_items',
       'item_complexity_score', 'queue_length', 'kitchen_utilization',
       'weather', 'festival_flag', 'weekend', 'prep_time',
       'restaurant_avg_prep', 'hour_sin', 'hour_cos', 'congestion_complexity',
       'log_queue'],
      dtype='str')

Final Shape: (550502, 18)


,cuisine_type,restaurant_capacity,restaurant_rating,chef_count,efficiency_score,total_items,item_complexity_score,queue_length,kitchen_utilization,weather,festival_flag,weekend,prep_time,restaurant_avg_prep,hour_sin,hour_cos,congestion_complexity,log_queue
0,0,10,3.6,5,0.876097,4,6.24,0.0,0.0,1,1,0,25.77,26.665551,0.0,1.0,0.0,0.0
1,3,13,3.0,8,0.812020,6,14.66,0.0,0.0,1,1,0,55.89,40.338372,0.0,1.0,0.0,0.0
2,5,5,4.5,3,0.994893,4,4.46,0.0,0.0,1,1,0,8.59,24.787594,0.0,1.0,0.0,0.0
3,0,7,4.1,4,0.978308,5,11.80,0.0,0.0,1,1,0,33.36,30.619106,0.0,1.0,0.0,0.0
4,6,5,4.1,3,0.980710,2,3.63,0.0,0.0,1,1,0,24.25,30.114836,0.0,1.0,0.0,0.0


In [13]:
# Separate target
X = df.drop(columns=["prep_time"])
y = df["prep_time"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (550502, 17)
Target shape: (550502,)


In [14]:
df.to_csv("zomato_kpt_feature_engineered.csv", index=False)
print("Feature engineered dataset saved successfully.")

Feature engineered dataset saved successfully.
